# Notebook 3: Overfitting (why the training score lies)

Step 3 of the ladder. Notebooks 1 and 2 trained models and reported how well they did **on the data they trained on**. This notebook shows why that number can't be trusted, and what to do about it. It is the most important idea in the whole series: every tuning knob you'll ever turn is judged against it.

You need: the loss-and-search picture from Notebook 1. NumPy and matplotlib only; runs in seconds.

### What you'll be able to say at the end
- **Overfitting** = the model learned the *noise* in the training set, not the pattern, so it does worse on new data.
- Why training error can only go down as a model grows, while error on new data goes down and then **back up**.
- Three ways to fight it: **more data**, **a smaller or leashed model** (regularization), and **stopping early**.
- The rule that protects every result you report: **never choose a knob on the data you score on.** Three separate piles: *train*, *validation*, *test*.

Same markers as before: `WHAT / WHY / SAME AS / BECOMES / WATCH` comments, **PREDICT** cells, a knob ledger, and a sandbox.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# WHAT: plot styling only (same quiet look as Notebooks 1 and 2). Nothing here affects the math.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, MUTED, GRIDC = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.titlecolor": INK, "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

def tag(ax, x, y, text, color, dx=4, dy=0, ha="left"):
    # WHAT: write a small label next to a line instead of making you decode a legend.
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9, va="center", ha=ha)

print("ready")

ready


# Part 1: A small, noisy data set, and a model that can bend as much as you like

> **You are here:** the setup for everything that follows.
> **The data:** a machine's measured drift `y` versus an operating reading `x`. The true relationship is a smooth wave plus a gentle slope. Every measurement has noise on top (sensors are never perfect). We only get **20** measurements to learn from.

**The model:** a **polynomial** of a chosen **degree**. Degree 1 is a straight line, degree 3 is a gentle curve with up to two bends, degree 14 can wiggle up and down many times. Higher degree means **more parameters** (degree + 1 of them), so more flexibility. The *degree* is a **model knob**, not something training learns.

(Under the hood, we fit each polynomial with the textbook least-squares solver instead of a gradient-descent loop, because it's instant and gives the same answer descent would reach. Notebook 1 showed that for a line, and the same holds here. We call the answer the "best fit".)

In [2]:
def truth(x):
    # WHAT: the real pattern the world follows (the model never sees this formula, only noisy samples of it)
    return np.sin(1.5 * x) + 0.3 * x

NOISE = 0.3          # WHAT: size of the random measurement noise. Its square (0.09) is the best ANY model can score: the "noise floor".

def make_data(n, seed, f=truth):
    r = np.random.default_rng(seed)
    x = r.uniform(-3, 3, n)
    return x, f(x) + r.normal(0, NOISE, n)

def features(x, degree):
    # WHAT: turn each x into [1, x, x^2, ... x^degree], using a numerically well-behaved version of those powers (Legendre).
    # WHY:  plain powers of x get hugely different in size at high degree, which makes the solver unstable. Same idea as Notebook 1's
    #       "scale your inputs". The curve you can build is exactly the same set of polynomials.
    return np.polynomial.legendre.legvander(x / 3.0, degree)

def fit_poly(x, y, degree):
    w, *_ = np.linalg.lstsq(features(x, degree), y, rcond=None)     # WHAT: the weights that minimize squared error on THIS data
    return w

def predict(w, x):
    return features(x, len(w) - 1) @ w

def mse(a, b):
    return float(np.mean((a - b) ** 2))

x_tr, y_tr = make_data(20, seed=1)        # WHAT: TRAIN set: the only data the model is allowed to learn from
x_va, y_va = make_data(30, seed=2)        # WHAT: VALIDATION set: used to choose knobs (Part 6). Never trained on.
x_te, y_te = make_data(1000, seed=3)      # WHAT: TEST set: used once, at the end, to report. Never trained on, never used to choose.
print(f"train {len(x_tr)} points, validation {len(x_va)}, test {len(x_te)}.   noise floor = {NOISE**2:.3f}")

train 20 points, validation 30, test 1000.   noise floor = 0.090


# Part 2: Three models, one data set

> **You are here:** fit degree 1, 3 and 14 to the *same* 20 points and look.

**PREDICT first.** Degree 14 has 15 parameters for 20 points, enough freedom to hit nearly every point. (a) What will its **training** error be: tiny, or about the noise floor? (b) What will its error on **new** points be: better than degree 3, the same, or worse? Type a guess, then run.

In [3]:
# PREDICT: degree-14 training error? and its error on NEW points compared with degree 3?
my_prediction = ""
print("your prediction:", my_prediction if my_prediction else "(blank: type one above first)")

your prediction: (blank: type one above first)


In [4]:
print(" degree | parameters | train error | validation error | test error")
for d in (1, 3, 14):
    w = fit_poly(x_tr, y_tr, d)
    print(f"  {d:4d}  |    {d + 1:4d}    |  {mse(predict(w, x_tr), y_tr):10.4f} | {mse(predict(w, x_va), y_va):16.4f} | {mse(predict(w, x_te), y_te):.4f}")
print(f"(noise floor, the best any model could do on new data: {NOISE**2:.3f})")

 degree | parameters | train error | validation error | test error
     1  |       2    |      0.6879 |           0.7578 | 0.6743
     3  |       4    |      0.1757 |           0.1523 | 0.1937
    14  |      15    |      0.0412 |          28.5157 | 928.9566
(noise floor, the best any model could do on new data: 0.090)


In [5]:
xs = np.linspace(-3, 3, 400)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, d, name in zip(axes, (1, 3, 14), ("too stiff", "about right", "too flexible")):
    w = fit_poly(x_tr, y_tr, d)
    ax.plot(xs, truth(xs), color="#898781", linestyle="--", linewidth=1.4)
    ax.scatter(x_tr, y_tr, s=22, color=INK, zorder=3)
    ax.plot(xs, predict(w, xs), color=BLUE if d == 3 else ORANGE)
    ax.set_ylim(-3, 3); ax.set_xlabel("reading x")
    ax.set_title(f"degree {d}: {name}")
axes[0].set_ylabel("measured drift y")
axes[0].annotate("dashed = the true pattern", (-2.9, -2.6), fontsize=9, color=MUTED)
plt.tight_layout(); plt.show()

<Figure size 1320x418 with 3 Axes>

**What to notice.**
- **Degree 1** is too stiff to follow the wave: it's wrong on the training data *and* on new data. That's **underfitting** (the model can't represent the pattern).
- **Degree 3** follows the true pattern closely. Its training error (0.18) is about twice the noise floor (0.09): a stiff 4-parameter model leaves some real misfit, but it isn't chasing noise.
- **Degree 14** passes almost exactly through the training points (tiny training error) but swings wildly between and beyond them. It learned each point's *noise* as if it were signal. Its error on new points is far worse. That's **overfitting**.

Training error rewarded degree 14. New-data error punished it. **Only the second one is what we care about.**

# Part 3: The U-curve

> **You are here:** you saw three degrees. Now sweep every degree from 0 to 14 and plot all three errors.

(One catch to keep in mind: the **validation** and **test** points are used here only to *score* each fit. None of them trains anything.)

In [6]:
degrees = list(range(0, 15))
tr_err, va_err, te_err = [], [], []
for d in degrees:
    w = fit_poly(x_tr, y_tr, d)
    tr_err.append(mse(predict(w, x_tr), y_tr)); va_err.append(mse(predict(w, x_va), y_va)); te_err.append(mse(predict(w, x_te), y_te))

print(" degree |  train   | validation |   test")
for d in degrees:
    print(f"  {d:4d}  | {tr_err[d]:8.3f} | {va_err[d]:10.3f} | {te_err[d]:9.3f}")
best_d_test = int(np.argmin(te_err)); best_d_val = int(np.argmin(va_err))
print()
print(f"train error is smallest at degree {int(np.argmin(tr_err))}  (it can only go down as the model grows)")
print(f"validation error is smallest at degree {best_d_val};  test error is smallest at degree {best_d_test}")

 degree |  train   | validation |   test
     0  |    1.044 |      1.167 |     0.965
     1  |    0.688 |      0.758 |     0.674
     2  |    0.642 |      0.785 |     0.695
     3  |    0.176 |      0.152 |     0.194
     4  |    0.148 |      0.143 |     0.221
     5  |    0.076 |      0.107 |     0.145
     6  |    0.067 |      0.184 |     0.234
     7  |    0.067 |      0.204 |     0.277
     8  |    0.062 |      0.391 |     1.107
     9  |    0.061 |      0.250 |     0.387
    10  |    0.056 |      0.573 |     5.016
    11  |    0.050 |      3.120 |    33.996
    12  |    0.047 |      5.370 |    38.474
    13  |    0.042 |     32.700 |   405.501
    14  |    0.041 |     28.516 |   928.957

train error is smallest at degree 14  (it can only go down as the model grows)
validation error is smallest at degree 5;  test error is smallest at degree 5


In [7]:
fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.plot(degrees, np.maximum(tr_err, 1e-4), color=BLUE, marker="o", markersize=4)
ax.plot(degrees, te_err, color=ORANGE, marker="o", markersize=4)
ax.axhline(NOISE ** 2, color="#898781", linestyle="--", linewidth=1.2)
ax.set_yscale("log"); ax.set_xlabel("model flexibility (polynomial degree)"); ax.set_ylabel("error (log scale)")
tag(ax, 14, max(tr_err[-1], 1e-4), "train", BLUE, dx=6); tag(ax, 14, te_err[-1], "test", ORANGE, dx=6)
tag(ax, 0.2, NOISE ** 2 * 0.72, "noise floor", MUTED)
ax.set_xlim(-0.5, 16.2); ax.set_title("Train error only falls. Test error falls, then climbs.")
plt.show()

<Figure size 814x440 with 1 Axes>

**What to notice.** This is the picture to remember.
- **Training error** (blue) falls as the model gets more flexible, all the way down. It can *never* tell you when to stop.
- **Test error** (orange) falls while the model is learning real pattern, bottoms out near the sweet spot, then **climbs** as it starts fitting noise. The gap between the two lines is the **overfitting gap**.
- **Test** error never gets below the **noise floor** (dashed); nothing can predict random noise on new data. But **training** error *does* drop below it from degree 5 on. That is the giveaway: a model scoring better than the noise floor on its own training data is fitting the noise.

The sweet spot depends on how much data you have and how noisy it is. It isn't a number you can look up.

# Part 4: Why it happens (bias and variance, measured)

> **You are here:** you know *that* big models overfit. Here is *why*, measured rather than argued.

Imagine collecting a fresh set of 20 noisy measurements, over and over, and fitting the same degree each time. Two things can go wrong with the fits:
- **Bias:** on average (across all those data sets) the fit is *off from the truth*, because the model is too simple to represent it.
- **Variance:** each individual fit **differs wildly** from the others, because it chases whatever noise that particular data set had.

Total expected error = bias² + variance + noise floor. Measure it: 300 fresh training sets per degree. (I stop at degree 7 on purpose: beyond that, a few wild fits dominate the 300-set averages and the bias/variance split stops being measurable. That wildness is itself the point.)

In [8]:
grid = np.linspace(-3, 3, 100)                         # WHAT: the places where we compare the fits with the truth
n_sets = 300
rows = []
for d in (0, 1, 2, 3, 4, 5, 6, 7):
    preds = np.array([predict(fit_poly(*make_data(20, 1000 + s), d), grid) for s in range(n_sets)])   # WHAT: one fit per fresh data set
    mean_fit = preds.mean(axis=0)                               # WHAT: the "average opinion" of all 300 fits
    bias2 = float(np.mean((mean_fit - truth(grid)) ** 2))       # WHAT: how far the AVERAGE fit is from the truth
    var = float(np.mean(preds.var(axis=0)))                     # WHAT: how much individual fits scatter around their own average
    rows.append((d, bias2, var))

print(" degree |  bias^2  | variance | bias^2 + variance + noise floor")
for d, b2, v in rows:
    print(f"  {d:4d}  | {b2:8.3f} | {v:8.3f} | {b2 + v + NOISE ** 2:12.3f}")

 degree |  bias^2  | variance | bias^2 + variance + noise floor
     0  |    0.737 |    0.042 |        0.869
     1  |    0.483 |    0.058 |        0.631
     2  |    0.490 |    0.156 |        0.736
     3  |    0.079 |    0.070 |        0.239
     4  |    0.113 |    0.287 |        0.490
     5  |    0.002 |    0.328 |        0.420
     6  |    0.010 |    0.991 |        1.091
     7  |    0.033 |    5.863 |        5.986


**What to notice.**
- Low degrees: **bias** is the big term (too stiff to follow the wave), variance is small. More flexibility cuts bias.
- Around degree 5 the *average* fit is almost exactly right (bias² near zero), but **variance** has already overtaken bias, and it keeps growing with flexibility: any single fit is at the mercy of its own noise. Past degree 7 (not shown) the individual fits get so wild that variance runs to the hundreds and beyond, which matches the enormous test errors in Part 3.
- The best degree balances the two. (Small bumps, like bias² rising slightly from degree 3 to 4, are measurement noise from using only 300 data sets; the overall trend is what matters.) This is the **bias-variance trade-off**, and the U-curve in Part 3 is its shadow.

This is also the reason overfitting is about **how much data you have relative to model size**: variance comes from fitting too many free parameters to too few noisy points.

# Part 5: Fix 1: more data

> **You are here:** variance comes from too few points for the parameters. So give it more points.

Same degree-3 and degree-12 models, but with 15, 40, 100 and 400 training points. Each cell is the **median** test error over 30 different training sets (the median keeps one unlucky data set from dominating).

In [9]:
sizes = [15, 40, 100, 400]
curve = {3: [], 12: []}
test_big = make_data(2000, seed=777)
for d in (3, 12):
    for n_ in sizes:
        errs = []
        for s in range(30):
            w = fit_poly(*make_data(n_, 5000 + s), d)
            errs.append(mse(predict(w, test_big[0]), test_big[1]))
        curve[d].append(float(np.median(errs)))

print(" training points |  degree 3 test error | degree 12 test error")
for i, n_ in enumerate(sizes):
    print(f"      {n_:4d}       |       {curve[3][i]:8.3f}       |       {curve[12][i]:8.3f}")
print(f"(noise floor {NOISE**2:.3f})")

 training points |  degree 3 test error | degree 12 test error
        15       |          0.228       |       491502.500
        40       |          0.174       |          0.416
       100       |          0.162       |          0.108
       400       |          0.157       |          0.093
(noise floor 0.090)


**What to notice.** Degree 12 is a disaster at 15 points, worse than degree 3 at 40, and at 100 and 400 points it **overtakes** degree 3 and approaches the noise floor. The flexible model needed enough data to *earn* its flexibility. The simple model was fine from the start but can't do better than its bias allows. So "more data" works, but it's usually the expensive fix: in RL it means more episodes, and in a factory it means more parts measured.

# Part 6: Fix 2: put the big model on a leash (regularization)

> **You are here:** you can't always get more data. Instead, keep the flexible model but **penalize big weights**.

Overfit curves swing wildly *because their weights get huge* (to hit every point). So add a cost for large weights to the loss:

`loss = (mean squared error) + lam * (sum of squared weights)`

`lam` is a new **knob** (the **regularization strength**): `lam = 0` is the unleashed model, large `lam` pins the weights near zero (a nearly flat curve). It's an *objective* knob, since it changes what "best" means. For this loss the best fit still has a direct formula, so the cell below uses it.

**But how do we choose `lam`?** Not by training error (it would always pick `lam = 0`). Not by test error (that's cheating, see Part 7). We use the **validation** set.

In [10]:
def fit_ridge(x, y, degree, lam):
    V = features(x, degree)
    pen = lam * np.eye(degree + 1); pen[0, 0] = 0.0          # WHAT: penalty matrix. We do NOT penalize the constant term (the overall height).
    # WHAT: the weights minimizing  mean((V w - y)^2) + lam * sum(w^2)   (the closed-form answer, like polyfit was for a line)
    return np.linalg.solve(V.T @ V / len(y) + pen, V.T @ y / len(y))

lams = [0.0] + list(np.logspace(-6, 1, 15))
r_tr, r_va, r_te = [], [], []
for lam in lams:
    w = fit_ridge(x_tr, y_tr, 14, lam)
    r_tr.append(mse(predict(w, x_tr), y_tr)); r_va.append(mse(predict(w, x_va), y_va)); r_te.append(mse(predict(w, x_te), y_te))

print("  lam       |  train  | validation |   test    | size of weights (sum of squares)")
for lam, a, b, c in zip(lams, r_tr, r_va, r_te):
    w = fit_ridge(x_tr, y_tr, 14, lam)
    print(f"  {lam:9.2g} | {a:7.3f} | {b:10.3f} | {c:9.3f} | {np.sum(w[1:] ** 2):10.3g}")
best_lam = lams[int(np.argmin(r_va))]
print()
print(f"validation picks lam = {best_lam:.2g}  ->  test error {r_te[int(np.argmin(r_va))]:.3f}   (unleashed degree 14: {r_te[0]:.3f}; best plain degree by validation: {te_err[best_d_val]:.3f})")

  lam       |  train  | validation |   test    | size of weights (sum of squares)
          0 |   0.041 |     28.516 |   928.957 |   1.19e+04
      1e-06 |   0.043 |     13.741 |   184.382 |   2.32e+03
    3.2e-06 |   0.046 |      5.734 |    60.235 |        759
      1e-05 |   0.049 |      1.941 |    11.526 |        150
    3.2e-05 |   0.051 |      0.913 |     2.008 |       30.5
     0.0001 |   0.052 |      0.562 |     0.649 |       12.6
    0.00032 |   0.052 |      0.357 |     0.369 |       8.44
      0.001 |   0.053 |      0.240 |     0.256 |       6.89
     0.0032 |   0.055 |      0.188 |     0.204 |       6.15
       0.01 |   0.060 |      0.177 |     0.181 |       5.28
      0.032 |   0.087 |      0.213 |     0.187 |       3.88


        0.1 |   0.193 |      0.342 |     0.266 |       2.11
       0.32 |   0.443 |      0.603 |     0.469 |        0.7
          1 |   0.739 |      0.886 |     0.712 |      0.136
        3.2 |   0.925 |      1.058 |     0.866 |     0.0181
         10 |   1.003 |      1.130 |     0.931 |    0.00201

validation picks lam = 0.01  ->  test error 0.181   (unleashed degree 14: 928.957; best plain degree by validation: 0.145)


In [11]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.9))
lx = np.array(lams[1:])
a1.plot(lx, r_tr[1:], color=BLUE); a1.plot(lx, r_va[1:], color=AQUA); a1.plot(lx, r_te[1:], color=ORANGE)
a1.axhline(NOISE ** 2, color="#898781", linestyle="--", linewidth=1.2)
a1.set_xscale("log"); a1.set_yscale("log"); a1.set_xlabel("regularization strength lam (log scale; leash tighter to the right)"); a1.set_ylabel("error (log scale)")
tag(a1, lx[-1], r_tr[-1], "train", BLUE, dx=5); tag(a1, lx[-1], r_va[-1], "validation", AQUA, dx=5, dy=8); tag(a1, lx[-1], r_te[-1], "test", ORANGE, dx=5, dy=-8)
a1.set_xlim(lx[0], lx[-1] * 30); a1.set_title("Degree 14 with a leash: same U-curve, new knob")
w_best = fit_ridge(x_tr, y_tr, 14, best_lam); w_none = fit_ridge(x_tr, y_tr, 14, 0.0)
a2.plot(xs, truth(xs), color="#898781", linestyle="--", linewidth=1.4)
a2.scatter(x_tr, y_tr, s=22, color=INK, zorder=3)
a2.plot(xs, predict(w_none, xs), color=ORANGE, label="lam = 0")
a2.plot(xs, predict(w_best, xs), color=BLUE, label=f"lam = {best_lam:.2g} (picked on validation)")
a2.set_ylim(-3, 3); a2.set_xlabel("reading x"); a2.set_title("Same degree-14 model, with and without the leash"); a2.legend(loc="lower right")
plt.tight_layout(); plt.show()

<Figure size 1265x429 with 2 Axes>

**What to notice.**
- The weights column shrinks as `lam` grows: that is the leash working. A tamer set of weights can't make the wild swings.
- Validation error is U-shaped in `lam` just as test error was in degree: too loose overfits, too tight underfits (high `lam` flattens the curve toward a line and then toward a constant).
- The leashed degree-14 model went from a test error of about 929 to about 0.18, close to (though in this draw not better than) the best plain-degree model. Regularization lets you use a flexible model *without* having to guess its size exactly. That's why neural networks, which have far more parameters than data points, are normally trained with some form of it.

(Two honest caveats: 30 validation points make the choice of `lam` a bit noisy, and I swept only 16 values, so the exact winner depends on this particular draw.)

# Part 7: The rule that protects you: three piles, and why you can't peek

> **You are here:** you've chosen knobs (degree, `lam`) by looking at scores. If you look at the *same* scores you report, you fool yourself.

Why: pick the best of 15 degrees by their score on some set of 30 points, and the winner is partly the one that *got lucky* on those 30 points. Its score there is optimistic. Measure how optimistic, over 300 repeated experiments:
- **Leaky:** choose the degree on a 30-point set and report the score *on that same set*.
- **Honest:** judge the chosen degree on a huge independent test set (its true quality).

In [12]:
rng_t = np.random.default_rng(99)
reported, true_quality, chosen = [], [], []
test_huge = make_data(3000, seed=4242)
for t in range(300):
    xt, yt = make_data(20, 20000 + 3 * t)              # train
    xs_, ys_ = make_data(30, 20001 + 3 * t)            # the 30-point set we choose on (and, if leaky, also report on)
    fits = [fit_poly(xt, yt, d) for d in range(15)]
    scores = [mse(predict(w, xs_), ys_) for w in fits]
    pick = int(np.argmin(scores))
    reported.append(scores[pick])                                              # leaky: what we'd have told people
    true_quality.append(mse(predict(fits[pick], test_huge[0]), test_huge[1]))   # honest: how good the pick really is
    chosen.append(pick)

print(f"average score we would REPORT after choosing on the same set : {np.mean(reported):.3f}")
print(f"average TRUE error of the chosen model (independent test set) : {np.mean(true_quality):.3f}")
print(f"optimism: the leaky number is {np.mean(true_quality) - np.mean(reported):.3f} too low, "
      f"i.e. it makes the model look {100 * (1 - np.mean(reported) / np.mean(true_quality)):.0f}% better than it is")
print(f"(for scale: the noise floor is {NOISE**2:.3f})")
print(f"degrees chosen across the 300 repeats: min {min(chosen)}, median {int(np.median(chosen))}, max {max(chosen)}")

average score we would REPORT after choosing on the same set : 0.145
average TRUE error of the chosen model (independent test set) : 0.189
optimism: the leaky number is 0.044 too low, i.e. it makes the model look 23% better than it is
(for scale: the noise floor is 0.090)
degrees chosen across the 300 repeats: min 0, median 5, max 12


**What to notice.** Choosing and scoring on the same data makes the number you report too good. The cure is **three separate piles**:

| Pile | Used for | May it influence choices? |
|---|---|---|
| **Train** | fitting the parameters (the weights) | yes, that's its job |
| **Validation** | choosing knobs (degree, `lam`, learning rate, when to stop) | yes |
| **Test** | the one honest score you report at the end | **never** |

If you look at the test score and then change a knob, the test set has become a validation set, and you need a new, untouched test set. This is also why in school assignments you shouldn't tune on the grader's numbers.

(Also notice the chosen degree varies a lot between repeats. With 20 noisy points, even the "best" model is partly luck. Honest numbers come with honest error bars.)

# Part 8: Fix 3: stop early (search as regularization)

> **You are here:** a third leash needs no penalty term: **stop training before the model has finished fitting.**

Start the degree-14 model at all-zero weights and train by **gradient descent** (Notebook 1's loop), watching both training and validation error each step. The model first picks up the large smooth pattern, then slowly bends to fit the noise. Stop when the *validation* error is lowest.

In [13]:
Vtr, Vva = features(x_tr, 14), features(x_va, 14)
n_tr = len(y_tr)
H_top = np.linalg.eigvalsh(2 * Vtr.T @ Vtr / n_tr).max()      # WHAT: the stiffest curvature of the loss (Notebook 1, Part 4)
lr_es = 1.0 / H_top                                           # WHAT: a safe step: half the speed limit 2/lambda_max
w = np.zeros(15)
steps_logged, tr_curve, va_curve = [], [], []
for step in range(0, 200001):
    if step % 100 == 0 or step < 100:
        steps_logged.append(step)
        tr_curve.append(mse(Vtr @ w, y_tr)); va_curve.append(mse(Vva @ w, y_va))
    grad = 2 * Vtr.T @ (Vtr @ w - y_tr) / n_tr                # SAME AS: Notebook 1's gradient for a mean-squared-error fit
    w = w - lr_es * grad                                       # SAME AS: Notebook 1's one-line update

steps_logged = np.array(steps_logged)
i_best = int(np.argmin(va_curve))
print(" step     | train error | validation error")
for s in (0, 10, 100, 1000, 10000, 100000, 200000):
    i = int(np.where(steps_logged == s)[0][0])
    print(f" {s:8d} | {tr_curve[i]:11.3f} | {va_curve[i]:16.3f}")
print()
print(f"validation error is lowest at step {steps_logged[i_best]} ({va_curve[i_best]:.3f}); at the end it is {va_curve[-1]:.3f}")

 step     | train error | validation error
        0 |       1.147 |            1.034
       10 |       0.107 |            0.244
      100 |       0.055 |            0.165
     1000 |       0.053 |            0.259
    10000 |       0.052 |            0.692
   100000 |       0.049 |            2.071
   200000 |       0.047 |            4.296

validation error is lowest at step 62 (0.163); at the end it is 4.296


In [14]:
fig, ax = plt.subplots(figsize=(7.4, 4.0))
ax.plot(steps_logged[1:], tr_curve[1:], color=BLUE); ax.plot(steps_logged[1:], va_curve[1:], color=AQUA)
ax.axvline(steps_logged[i_best], color=ORANGE, linestyle="--", linewidth=1.3)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("gradient-descent step (log scale)"); ax.set_ylabel("error (log scale)")
tag(ax, steps_logged[-1], tr_curve[-1], "train", BLUE, dx=6); tag(ax, steps_logged[-1], va_curve[-1], "validation", AQUA, dx=6)
tag(ax, steps_logged[i_best], ax.get_ylim()[1] * 0.6, "stop here", ORANGE, dx=-5, ha="right")
ax.set_xlim(1, 2.5e6); ax.set_title("Train error keeps falling; validation turns around")
plt.show()

<Figure size 814x440 with 1 Axes>

**What to notice.** The training error keeps falling for as long as you let it run, while the validation error turns around at some step. The step to stop at is one more **judging** knob from the Notebook 1 ledger (the stopping rule), and now you can see what it's protecting you from. This is the standard way large neural networks are kept from overfitting: they train and watch a validation score, and keep the best checkpoint.

(One honest limit: this single run is on one train/validation draw, so the exact best step is noisy; the *shape* is what generalizes.)

# Part 9: What this means for reinforcement learning

> **You are here:** tying the idea back to Qmaze and CartPole.

Overfitting isn't only a regression problem. In RL it shows up as:
- **Memorizing the maze, not learning to navigate.** A Q-table or network trained on one maze layout, with one set of starting squares, can look perfect there and fail on a different layout or start. The Qmaze notebook's final "solves every reachable square" check reused the same squares it had trained from, so it was a *training-set* score. An honest test would use a **held-out maze**.
- **Overfitting to the replay buffer.** DQN trains on a stored batch of past moves; a big network can fit the quirks of that batch.
- **Seeds as the test set.** A CartPole agent that scores 500 on the seeds you tuned on may not on fresh seeds. Re-run on seeds you never tuned on (the RL version of a test pile).
- **The same three leashes exist:** more experience (more episodes), regularization (weight decay, smaller networks, entropy bonuses that stop the policy from collapsing too early), and early stopping (keep the best checkpoint by validation score).

Nothing here is new machinery. It's the same train/validation/test discipline, applied to episodes instead of rows.

# Part 10: Sandbox (your ideas compete here)

> **You are here:** pick a model, and see whether it generalizes across three different worlds.

**The ruler.** A model is a function `model(x_train, y_train, x_eval)` returning predictions at `x_eval`. It sees **only the 20 training points**, so any choosing of knobs inside must use only those (for instance, cross-validation: split the training points into folds). It's scored on 1000 fresh test points, **taken as the median over 12 different training sets** so luck (and one wild fit) doesn't decide. Three worlds: the **smooth wave** from this notebook, a **faster wave**, and a **nearly straight line**.

In [15]:
worlds = [("smooth wave", truth),
          ("faster wave", lambda x: np.sin(3.0 * x)),
          ("nearly straight", lambda x: 0.5 * x)]
n_draws = 12

def kfold_pick(x, y, candidates, fit_fn, k=5):
    # WHAT: choose the candidate knob value with the lowest k-fold cross-validation error, using ONLY the training points.
    # WHY:  cross-validation = rotate which fifth of the training data plays "validation". No test data touched.
    idx = np.arange(len(y)); folds = np.array_split(np.random.default_rng(0).permutation(idx), k)
    best, best_err = None, np.inf
    for c in candidates:
        errs = []
        for f in folds:
            tr = np.setdiff1d(idx, f)
            errs.append(mse(predict(fit_fn(x[tr], y[tr], c), x[f]), y[f]))
        if np.mean(errs) < best_err: best, best_err = c, np.mean(errs)
    return best

def score_model(model):
    res = []
    for _, f in worlds:
        errs = []
        for s in range(n_draws):
            x, y = make_data(20, 90000 + s, f)
            xe, ye = make_data(1000, 91000 + s, f)
            errs.append(mse(model(x, y, xe), ye))
        res.append(float(np.median(errs)))
    return res

def race3(entries):
    print(f"{'':44s}" + "".join(f"{nm:>17s}" for nm, _ in worlds) + f"{'average':>12s}")
    for label, model in entries:
        r = score_model(model)
        print(f"{label:44s}" + "".join(f"{v:17.3f}" for v in r) + f"{np.mean(r):12.3f}")

def fixed_degree(d):
    return lambda x, y, xe: predict(fit_poly(x, y, d), xe)

def cv_degree(x, y, xe):
    d = kfold_pick(x, y, list(range(0, 11)), lambda xx, yy, c: fit_poly(xx, yy, c))
    return predict(fit_poly(x, y, d), xe)

def cv_ridge(x, y, xe):
    lam = kfold_pick(x, y, [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0], lambda xx, yy, c: fit_ridge(xx, yy, 14, c))
    return predict(fit_ridge(x, y, 14, lam), xe)

reference3 = [("degree 1 (stiff)", fixed_degree(1)), ("degree 3", fixed_degree(3)), ("degree 14, no leash", fixed_degree(14)),
              ("degree picked by 5-fold CV", cv_degree), ("degree 14 + leash picked by 5-fold CV", cv_ridge)]
race3(reference3)
print(f"(noise floor {NOISE**2:.3f}: nobody can beat it)")

                                                  smooth wave      faster wave  nearly straight     average
degree 1 (stiff)                                        0.598            0.630            0.099       0.442
degree 3                                                0.219            0.559            0.108       0.295
degree 14, no leash                                  3727.375         3821.819         3727.368    3758.854


degree picked by 5-fold CV                              0.207            0.651            0.102       0.320


degree 14 + leash picked by 5-fold CV                   0.169            0.223            0.167       0.186
(noise floor 0.090: nobody can beat it)


**What to notice.**
- No fixed degree wins every world: degree 1 is best on the nearly straight line (0.099, almost the noise floor), degree 3 beats it on the smooth wave, and neither follows the faster wave well.
- **Degree 14 with no leash is catastrophic everywhere** (errors in the thousands), even as a median over 12 draws. With 20 points, a model that flexible is unusable unless something restrains it.
- **Degree 14 with a leash picked by cross-validation** has the best average and the best score on both wavy worlds, but it pays on the straight line (0.167 vs 0.099 for plain degree 1). It hedges: decent everywhere, best nowhere simple.
- **Degree picked by cross-validation** tracks the best fixed degree on the smooth and straight worlds but does poorly on the faster wave (0.651). I haven't dug into why; with only 20 points, cross-validation itself is noisy.
- What is *not* tested: all three worlds are smooth, all use 20 points and the same noise level, so other settings could reorder things. The "average" column weights the three worlds equally, which is my choice, not a law.

### Your slot
Edit `my_model` and re-run. Ideas: other degrees; a different set of `lam` candidates; a model that **averages** several degrees' predictions; Gaussian-bump or sine features instead of polynomials; a decision rule like "use degree 1 unless CV says a higher degree is clearly better". Rule of the game: use **only the training points** to decide anything.

In [16]:
def my_model(x, y, xe):
    # SANDBOX: replace with your own idea. You get 20 training points and must return predictions at xe. No peeking at test data.
    w = fit_poly(x, y, 3)             # <- starts as degree 3. Change it, or choose the degree with kfold_pick(...)
    return predict(w, xe)

race3(reference3 + [("MY MODEL", my_model)])

                                                  smooth wave      faster wave  nearly straight     average
degree 1 (stiff)                                        0.598            0.630            0.099       0.442
degree 3                                                0.219            0.559            0.108       0.295
degree 14, no leash                                  3727.375         3821.819         3727.368    3758.854


degree picked by 5-fold CV                              0.207            0.651            0.102       0.320


degree 14 + leash picked by 5-fold CV                   0.169            0.223            0.167       0.186
MY MODEL                                                0.219            0.559            0.108       0.295


# Part 11: The knob ledger, self-check, and what's next

## Knob ledger (Notebooks 1 to 3)
Families: **objective**, **model**, **search**, **experience**, **judging**. New in this notebook: model size, regularization, data amount, the three piles, and the stopping point.

In [17]:
rows = [
    ("learning rate (NB1)", "search", "too small: slow", "too big: blows up"),
    ("input scaling (NB1)", "experience", "unscaled: far slower", "standardized: fast and safe"),
    ("loss for categories (NB2)", "objective", "squared error stalls when wrong", "cross-entropy keeps a slope"),
    ("decision threshold (NB2)", "judging", "0.5 ignores costs", "set by the cost of each mistake"),
    ("model size (degree)", "model", f"degree 1: test {te_err[1]:.2f} (underfits)", f"degree 14: test {te_err[14]:.2f} (overfits)"),
    ("regularization strength lam", "objective", f"lam = 0: test {r_te[0]:.2f}", f"lam = {best_lam:.2g} (chosen on validation): test {r_te[int(np.argmin(r_va))]:.2f}"),
    ("amount of data", "experience", f"15 points, degree 12: median test {curve[12][0]:.2f}", f"400 points: {curve[12][-1]:.2f}"),
    ("stopping step", "judging", f"step 0 or too early: train/val both high", f"step {steps_logged[i_best]}: validation lowest; later it rises"),
    ("train / validation / test split", "judging", f"choosing on the report set: ~{100 * (1 - np.mean(reported) / np.mean(true_quality)):.0f}% too optimistic", "separate piles: honest number"),
]
print(f"{'knob':32s}{'family':12s}{'at one extreme':50s}{'at the other'}")
print("-" * 165)
for knob, fam, a, b in rows:
    print(f"{knob:32s}{fam:12s}{a:50s}{b}")

knob                            family      at one extreme                                    at the other
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
learning rate (NB1)             search      too small: slow                                   too big: blows up
input scaling (NB1)             experience  unscaled: far slower                              standardized: fast and safe
loss for categories (NB2)       objective   squared error stalls when wrong                   cross-entropy keeps a slope
decision threshold (NB2)        judging     0.5 ignores costs                                 set by the cost of each mistake
model size (degree)             model       degree 1: test 0.67 (underfits)                   degree 14: test 928.96 (overfits)
regularization strength lam     objective   lam = 0: test 928.96                              lam = 0.01 (chosen on v

## Self-check (answer in your own words before you read the answers)
| Question | Short answer |
|---|---|
| Why can't training error tell you the model is good? | A flexible enough model can drive it to nearly zero by memorizing noise. It only ever goes down as the model grows. |
| What is overfitting in one sentence? | The model learned the noise in the training set as if it were pattern, so it does worse on new data. |
| What's the difference between bias and variance? | Bias: the model is too simple, wrong on average. Variance: the model is so flexible that each fit chases its own data set's noise. |
| Name three fixes. | More data; a leash on the weights (regularization) or a smaller model; stop training early. |
| Why three data piles, not two? | Choosing a knob on a set makes that set's score optimistic. Test must stay untouched so the final number is honest. |
| How does this apply to the Qmaze check? | It reused its training squares, so it was a training-set score. A held-out maze would be the honest test. |

## What's next
- You can now distrust a training score on principle, and you know the three fixes and the three piles.
- Every knob in the ledger has a failure at each end. Tuning means finding the middle *using validation data, never test data*.

**Notebook 4: a neural network from scratch in NumPy.** Stack neurons from Notebook 2 so the network learns its own inputs (fixing the XOR failure). You'll write the forward pass, derive the backward pass with the chain rule, check it against nudging (as we did here), and then watch the same overfitting patterns appear in a network.